<h3>Final Refit and Single Held-Out Evaluation</h3>


<h3>1. Runtime Setup and Artifact Paths</h3>

<p>The next cell follows the model-training notebook's Colab setup: mount Drive, load the same environment-file locations, and locate or clone the configured repository branch. Colab inputs are:</p>

<p>MyDrive/capstone-data/feature_engineering/: the supervised H1 table and saved fold definitions.</p>

<p>MyDrive/capstone-data/model_training/: tuning_h1.json, including feature order, RF/XGBoost parameters, and Ridge alpha.</p>

<p>Local execution uses the equivalent results/artifacts/ directories. Output directories and the H1 configuration are defined here, but no training or test inference is performed.</p>

In [1]:
import base64
import os
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/RAM-AS-RAP/ucs35-mudibo-ml.git'
REPO_BRANCH = 'feature/model-training'

if IN_COLAB:
    drive.mount('/content/drive')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'pandas<3', 'numpy', 'scikit-learn', 'xgboost', 'joblib', 'python-dotenv',
    ])
    from dotenv import load_dotenv
    ENV_PATHS = [
        Path('/content/drive/MyDrive/capstone-data/.env'),
        Path('/content/drive/MyDrive/AgriSence/.env'),
        Path('/content/drive/MyDrive/.env'),
        Path('/content/.env'),
    ]
    for env_path in ENV_PATHS:
        if env_path.exists():
            load_dotenv(env_path, override=False)
    repo_dir = Path('/content/AgriSence_model_training')
    repo_candidates = [repo_dir]
else:
    repo_dir = None
    repo_candidates = [Path.cwd(), *Path.cwd().resolve().parents]

repo_root = next((
    path.resolve() for path in repo_candidates
    if (path / 'pyproject.toml').exists() and (path / 'src/agrisence').exists()
), None)
if repo_root is None and IN_COLAB:
    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    if not github_token:
        from google.colab import userdata
        for secret_name in ('GITHUB_TOKEN', 'GH_TOKEN'):
            try:
                github_token = userdata.get(secret_name)
            except Exception:
                continue
            if github_token:
                break
    if not github_token:
        raise RuntimeError(
            'Private repository clone requires GITHUB_TOKEN or GH_TOKEN. '
            'Use Colab Secrets with notebook access enabled, or a Drive .env file.'
        )
    auth_header = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run([
        'git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_header}',
        'clone', '-b', REPO_BRANCH, REPO_URL, str(repo_dir),
    ], check=True)
    del github_token, auth_header
    repo_root = repo_dir.resolve()
if repo_root is None:
    raise FileNotFoundError('Start this notebook from inside the AgriSence repository.')
if IN_COLAB:
    os.chdir(repo_root)
    load_dotenv(repo_root / '.env', override=False)

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(repo_root / 'src'))

if IN_COLAB:
    feature_dir = Path('/content/drive/MyDrive/capstone-data/feature_engineering')
    tuning_dir = Path('/content/drive/MyDrive/capstone-data/model_training')
else:
    feature_dir = repo_root / 'results/artifacts/feature_engineering'
    tuning_dir = repo_root / 'results/artifacts/model_training'

output_dir = repo_root / 'results/artifacts'
model_dir = output_dir / 'models'
drive_output_dir = Path('/content/drive/MyDrive/capstone-data/final_evaluation') if IN_COLAB else None
drive_model_dir = Path('/content/drive/MyDrive/capstone-data/models') if IN_COLAB else None
horizon_configs = {
    horizon: {
        'data_path': feature_dir / f'model_df_{horizon}.csv',
        'fold_path': feature_dir / f'folds_{horizon}.json',
        'tuning_path': tuning_dir / f'tuning_{horizon}.json',
        'target_col': f'target_{horizon}',
    }
    for horizon in ('h1',)
}
missing = [str(config[key]) for config in horizon_configs.values()
           for key in ('data_path', 'fold_path', 'tuning_path') if not config[key].exists()]
if missing:
    raise FileNotFoundError(
        'Task 1/2 artifacts are required. Confirm their saved Drive outputs:\n'
        + '\n'.join(missing)
    )
for directory in (output_dir, model_dir, drive_output_dir, drive_model_dir):
    if directory is not None:
        directory.mkdir(parents=True, exist_ok=True)
print(f'Repository: {repo_root}')
print(f'Active branch target: {REPO_BRANCH}')
print(f'Feature/fold inputs: {feature_dir}')
print(f'Frozen tuning inputs: {tuning_dir}')
print(f'Local results: {output_dir}; models: {model_dir}')
if IN_COLAB:
    print(f'Drive results: {drive_output_dir}; Drive models: {drive_model_dir}')

Mounted at /content/drive
Repository: /content/AgriSence_model_training
Active branch target: feature/model-training
Feature/fold inputs: /content/drive/MyDrive/capstone-data/feature_engineering
Frozen tuning inputs: /content/drive/MyDrive/capstone-data/model_training
Local results: /content/AgriSence_model_training/results/artifacts; models: /content/AgriSence_model_training/results/artifacts/models
Drive results: /content/drive/MyDrive/capstone-data/final_evaluation; Drive models: /content/drive/MyDrive/capstone-data/models


In [ ]:
import hashlib
from pathlib import Path

model_dir = Path('/content/drive/MyDrive/capstone-data/models')
for model_name in ('stacking_h1.pkl',):
    model_path = model_dir / model_name
    if not model_path.is_file():
        raise FileNotFoundError(f'Model file not found in Drive: {model_path}')
    print(model_name, hashlib.sha256(model_path.read_bytes()).hexdigest())

stacking_h1.pkl 07756dbeda51d0a678cfcefbdf0ad46741cb0e74864a835f3ac44016ea7e3b85
stacking_h2.pkl a776884f0f0100030cc4a059b8b25aea3cfb87f2ea236ac16dc675cc99c60493


In [ ]:
import joblib, hashlib, numpy as np, pandas as pd, sklearn, xgboost as xgb
from pathlib import Path

SRC = Path("/content/drive/MyDrive/capstone-data/models")   # folder with the original pickles
OUT = SRC / "repackaged"; OUT.mkdir(exist_ok=True)
DF_PATHS = {"h1": "/content/drive/MyDrive/capstone-data/feature_engineering/model_df_h1.csv"}
print("sklearn", sklearn.__version__, "xgboost", xgb.__version__)

for h in ["h1"]:
    b = joblib.load(SRC / f"stacking_{h}.pkl")
    cols = b["feature_cols"]

    # 1. XGBoost model in its portable JSON format
    b["xgb_model"].save_model(str(OUT / f"xgb_{h}.json"))

    # 2. Everything else the inference pipeline needs; the linear baseline is not needed
    rest = {k: b[k] for k in ["rf_model", "ridge_meta", "feature_cols", "target_col", "horizon"]}
    joblib.dump(rest, OUT / f"stack_{h}_sklearn.joblib")

    # 3. Golden predictions from the original bundles, for a parity test
    X = pd.read_csv(DF_PATHS[h])[cols].astype("float64").sample(30, random_state=0)
    rf = b["rf_model"].predict(X); xg = b["xgb_model"].predict(X)
    st = b["ridge_meta"].predict(np.column_stack([rf, xg]))
    g = X.copy(); g["rf_pred"] = rf; g["xgb_pred"] = xg; g["stack_pred"] = st
    g.to_csv(OUT / f"golden_{h}.csv", index=False)

    # 4. Round trip check: the JSON model must reproduce the original XGBoost predictions
    r = xgb.XGBRegressor(); r.load_model(str(OUT / f"xgb_{h}.json"))
    d = np.abs(r.predict(X) - xg).max()
    print(h, "max abs diff after JSON round trip:", d)
    assert d < 1e-6, "JSON round trip changed the predictions"

for p in sorted(OUT.iterdir()):
    print(p.name, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())

sklearn 1.6.1 xgboost 3.4.1


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


h1 max abs diff after JSON round trip: 0.0
h2 max abs diff after JSON round trip: 0.0
golden_h1.csv 6765 f6e37f0eacfafa16c4eba740717b7f682698cdcce82ffcde2c15297ef6817e62
golden_h2.csv 6693 a7dd67578d39ccd2573a361a9a0c1d21d28900754d245dc058ae8dc366468bc7
stack_h1_sklearn.joblib 1420290 cee15646bc305df5f99dc9beea1483a9f94bd082bfb2f4b6ba607adf6480b93b
stack_h2_sklearn.joblib 962898 a2e9c8735b521302140298804f3b2eeeb4b175f090b42aadef53b05c712c582f
xgb_h1.json 232355 aac43fa3c71f8e5c4dcce4b7de3c94ff65a97690e1cd6ae5bb5ee1215778378b
xgb_h2.json 232411 e61191ed84dd78929ba1799b5b269097711b3964e4d13f195cc4970142f9f3c9


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [3]:
import joblib
for h in ["h1"]:
    d = joblib.load(f"/content/drive/MyDrive/capstone-data/models/repackaged/stack_{h}_sklearn.joblib")
    print(h, list(d["ridge_meta"].feature_names_in_), d["ridge_meta"].coef_)

h1 ['rf_oof_pred', 'xgb_oof_pred'] [0.37859443 0.45444641]


<h3>2. Import the Shared Evaluation Workflow</h3>



In [4]:
import importlib
import sys

if globals().get('IN_COLAB', False):
    current_branch = subprocess.run(
        ['git', '-C', str(repo_root), 'branch', '--show-current'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current_branch != REPO_BRANCH:
        raise RuntimeError(f'Colab checkout is on {current_branch!r}; expected {REPO_BRANCH!r}.')
    tracked_changes = subprocess.run(
        ['git', '-C', str(repo_root), 'status', '--porcelain', '--untracked-files=no'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if tracked_changes:
        raise RuntimeError('Colab checkout has tracked edits. Preserve them before updating the repository.')

    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    if not github_token:
        from google.colab import userdata
        for secret_name in ('GITHUB_TOKEN', 'GH_TOKEN'):
            try:
                github_token = userdata.get(secret_name)
            except Exception:
                continue
            if github_token:
                break
    if not github_token:
        raise RuntimeError('Enable GITHUB_TOKEN or GH_TOKEN in Colab Secrets to update the private repository.')
    git_env = os.environ.copy()
    config_index = int(git_env.get('GIT_CONFIG_COUNT', '0'))
    git_env['GIT_CONFIG_COUNT'] = str(config_index + 1)
    git_env[f'GIT_CONFIG_KEY_{config_index}'] = 'http.https://github.com/.extraheader'
    git_env[f'GIT_CONFIG_VALUE_{config_index}'] = (
        'AUTHORIZATION: basic ' + base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    )
    try:
        subprocess.run(
            ['git', '-C', str(repo_root), 'fetch', 'origin', REPO_BRANCH],
            env=git_env, check=True,
        )
        subprocess.run(
            ['git', '-C', str(repo_root), 'merge', '--ff-only', f'origin/{REPO_BRANCH}'],
            check=True,
        )
    finally:
        del github_token, git_env
    revision = subprocess.run(
        ['git', '-C', str(repo_root), 'rev-parse', '--short', 'HEAD'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    print(f'Colab checkout updated: {REPO_BRANCH} at {revision}')
    for module_name in list(sys.modules):
        if module_name == 'agrisence' or module_name.startswith('agrisence.'):
            del sys.modules[module_name]

importlib.invalidate_caches()
try:
    from agrisence.training.evaluate import rescore_saved_predictions, run_final_evaluation
except ImportError as error:
    raise RuntimeError(
        f'Task 3 could not be imported from {repo_root}. '
        f'Confirm the commit defining run_final_evaluation was pushed to {REPO_BRANCH}. '
        f'Original import error: {error}'
    ) from error

Colab checkout updated: feature/model-training at 71958a5


RuntimeError: Task 3 could not be imported from /content/AgriSence_model_training. Confirm the commit defining run_final_evaluation was pushed to feature/model-training. Original import error: cannot import name 'rescore_saved_predictions' from 'agrisence.training.evaluate' (/content/AgriSence_model_training/src/agrisence/training/evaluate.py)

<h3>3. Configure Evaluation Outputs</h3>

In [3]:
evaluation_outputs = {
    'output_dir': output_dir,
    'model_dir': model_dir,
    'drive_output_dir': drive_output_dir,
    'drive_model_dir': drive_model_dir,
}

<h3>4. Final Evaluation: H1</h3>

<p>Runs once only. It refits on development rows, closes the held-out test, predicts it a single time, and saves two pooled tables for Linear Regression and the Stacking ensemble: log-change metrics (MAE, MSE, RMSE, R2) and reconstructed-price metrics (MAE, MSE, RMSE, MAPE). If the test is already closed this cell stops with an error; skip it and use the next section.</p>

In [ ]:
results_h1 = run_final_evaluation('h1', horizon_configs['h1'], **evaluation_outputs)
for scale, table in results_h1.items():
    print(f'H1 pooled {scale} metrics:')
    display(table)

<h3>5. Re-score Saved Predictions</h3>

<p>Recomputes both pooled results tables from the saved test_predictions_h1.csv. It does not load a model, refit, or predict, and it leaves the closure marker untouched, so it is safe to run after the test is closed. It writes results_h1_log_change.csv and results_h1_price.csv and refreshes the manifest's metric metadata.</p>

In [2]:
rescored_h1 = rescore_saved_predictions('h1', output_dir, drive_output_dir)
for scale, table in rescored_h1.items():
    print(f'H1 pooled {scale} metrics (from saved predictions):')
    display(table)

NameError: name 'rescore_saved_predictions' is not defined

<h3>6. Save and Verify the Fitted Models</h3>

<p>The evaluation workflow already saves the fitted models automatically. This explicit save-only step reloads the trusted H1 bundle from local storage (or Drive if the local copy is missing), checks that their estimators are fitted, and saves/verifies both destinations (local and Drive) without refitting or making predictions. It is safe to rerun independently of the closed test evaluations.</p>

<p>The bundle contains the fitted Random Forest, XGBoost, Ridge meta-model, linear-regression pipeline, horizon, and feature-column order.</p>

<p>Local: results/artifacts/models/stacking_h1.pkl.</p>

<p>Drive: MyDrive/capstone-data/models/stacking_h1.pkl.</p>

In [6]:
import joblib
from sklearn.utils.validation import check_is_fitted

saved_models = []
for horizon in ('h1',):
    model_filename = f'stacking_{horizon}.pkl'
    source_candidates = [model_dir / model_filename]
    if drive_model_dir is not None:
        source_candidates.append(drive_model_dir / model_filename)
    source_path = next((path for path in source_candidates if path.exists()), None)
    if source_path is None:
        raise FileNotFoundError(
            f'No saved {horizon.upper()} model bundle was found. '
            'Check the evaluation output paths; do not rerun a closed test evaluation.'
        )

    model_bundle = joblib.load(source_path)
    required_keys = {'rf_model', 'xgb_model', 'ridge_meta', 'linear_model', 'feature_cols', 'horizon'}
    missing_keys = required_keys.difference(model_bundle)
    if missing_keys or model_bundle['horizon'] != horizon:
        raise ValueError(f'Invalid {horizon.upper()} model bundle: missing keys={sorted(missing_keys)}')
    if not model_bundle['feature_cols']:
        raise ValueError(f'{horizon.upper()} feature order is missing.')
    for model_key in ('rf_model', 'xgb_model', 'ridge_meta', 'linear_model'):
        check_is_fitted(model_bundle[model_key])

    for destination, directory in (('Local', model_dir), ('Google Drive', drive_model_dir)):
        if directory is not None:
            directory.mkdir(parents=True, exist_ok=True)
            saved_path = directory / model_filename
            joblib.dump(model_bundle, saved_path)
            verified_bundle = joblib.load(saved_path)
            if verified_bundle['feature_cols'] != model_bundle['feature_cols']:
                raise ValueError(f'Feature order changed while saving {saved_path}')
            saved_models.append({
                'Horizon': horizon.upper(),
                'Destination': destination,
                'Model bundle': str(saved_path),
                'Features': len(verified_bundle['feature_cols']),
                'Size (bytes)': saved_path.stat().st_size,
            })

with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(saved_models))

,Horizon,Destination,Model bundle,Features,Size (bytes)
0,H1,Local,/content/AgriSence_model_training/results/artifacts/models/stacking_h1.pkl,14,1666351
1,H1,Google Drive,/content/drive/MyDrive/capstone-data/models/stacking_h1.pkl,14,1666351
2,H2,Local,/content/AgriSence_model_training/results/artifacts/models/stacking_h2.pkl,14,1209311
3,H2,Google Drive,/content/drive/MyDrive/capstone-data/models/stacking_h2.pkl,14,1209311
